# MetaHarmonizer quickstart

The shortest path from *your* messy data to mapped results. Unlike the other
notebooks in this folder (which measure accuracy against known labels), this one
runs in **`prod` mode**: you supply only the input, and the engine fills in the
answers — no ground truth required.

Two parts, run independently:

1. **OntologyMapper** — map free-text values to ontology terms.
2. **SchemaMapper** — map a data file's column names to standard field names.

For the full input/output reference, see
[`docs/input_formats.md`](../docs/input_formats.md). Run this notebook from this
`examples/` directory (the Jupyter default here) so the `data/` paths resolve.

## Setup

Install the package (`pip install -e .` from the repo root) first. In Jupyter,
`nest_asyncio` lets the engines' async corpus/API calls run inside the notebook
event loop — install with `pip install -e ".[notebook]"`.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import pandas as pd

## Part 1 — OntologyMapper: map free-text values to ontology terms

**Input:** just a list of strings — the terms you want to map. These are
deliberately messy (abbreviations, ALL CAPS, British spelling) to show the
embedding stages at work; perfectly canonical strings would short-circuit at
the exact-match stage.

> **Prerequisite — `UMLS_API_KEY` is required.** On the first run for a given
> NCIt corpus, the engine builds per-term *concept tables* from the NCI API (and
> downloads the sap-bert encoder, ~440 MB). Set `UMLS_API_KEY` in your `.env`
> first (see [`docs/input_formats.md`](../docs/input_formats.md) §3). Once built,
> the tables and FAISS index are cached, so later runs are fast.
>
> To keep this demo quick, the next cell uses a **smaller slice** of the bundled
> disease corpus (so the concept-table build stays short) that still contains the
> right targets for our three terms. For real use, pass your full `corpus_df` —
> or omit `corpus_df` entirely to fetch the full NCIt corpus (~4 min to build on
> first run).

In [ ]:
from metaharmonizer import OntoMapEngine

# The terms you want to map — this is the only required input in prod mode.
my_terms = [
    "SQUAMOUS CELL CARCINOMA, PHARYNX",
    "astrocytoma grade III-IV",
    "oesophageal tumour",   # British spelling, normalized automatically
]

# Small, fast corpus slice: rows of the bundled disease corpus relevant to our
# three terms, plus a few distractors. Real ontology labels + codes, but only a
# few concept tables to build. For real use: pass your full corpus_df, or omit
# corpus_df to fetch the full NCIt corpus.
full_corpus = pd.read_csv("data/disease_corpus_updated.csv")
relevant = full_corpus["label"].str.contains(
    "pharyn|astrocyt|esophag|oesophag", case=False, na=False
)
corpus_df = (
    pd.concat([full_corpus[relevant], full_corpus.sample(n=30, random_state=0)])
    .drop_duplicates("obo_id")
    .reset_index(drop=True)
)
print(f"corpus slice: {len(corpus_df)} terms")

In [ ]:
engine = OntoMapEngine(
    category="disease",
    query=my_terms,
    corpus_df=corpus_df,
    s2_method="sap-bert",   # biomedical sentence encoder
    s2_strategy="st",       # 'st' = SentenceTransformer mean pooling
    # no ground_truth_map → prod mode is inferred automatically
)

# First run builds concept tables + a FAISS index for this corpus and downloads
# the encoder; later runs reuse the cache and are fast.
results = engine.run()
results

**Reading the output** — one row per input term:

| Column | Meaning |
|--------|---------|
| `query` | your original term |
| `match1` … `match5` | top-5 candidate ontology labels, best first |
| `match1_score` … | similarity score for each candidate |
| `stage` | which stage produced the row (2 = embedding) |
| `ref_match` | `"Not Found"` in prod mode — it's the eval column; **ignore it here** |

To map your own values, replace `my_terms` with your list. To use the live
NCIt corpus (richer than the sample) drop `corpus_df=` and set `UMLS_API_KEY`.

In [ ]:
# The top mapping for each term, with its score:
results[["query", "match1", "match1_score"]]

## Part 2 — SchemaMapper: map column names to standard fields

**Input:** a path to a CSV/TSV. The **column names** are what gets mapped; a few
rows of values help the value- and numeric-matching stages. `mode="manual"`
runs Stages 1–3 and needs no LLM key (use `mode="auto"` + `GEMINI_API_KEY` to
add the Stage-4 LLM fallback).

On first use this downloads the `all-MiniLM-L6-v2` encoder (~80 MB).

In [ ]:
# Peek at the input — arbitrary column names to be standardized.
pd.read_csv("data/ucec_cptac_2020_before_harmonization.csv").head()

In [ ]:
from metaharmonizer import SchemaMapEngine

engine = SchemaMapEngine(
    input_path="data/ucec_cptac_2020_before_harmonization.csv",   # .csv read comma-separated; .tsv would be tab-separated
    mode="manual",   # Stages 1–3, no LLM key required
    top_k=5,
)

sm_results = engine.run_schema_mapping()
sm_results

**Reading the output** — one row per input column:

| Column | Meaning |
|--------|---------|
| `query` | your original column name |
| `stage` / `method` | which stage and matcher produced the mapping |
| `match1` … | top-k standard field names, best first |
| `match1_score` / `match1_source` | score and origin of each candidate |

Results are also written to a timestamped CSV under `SM_OUTPUT_DIR`
(default `~/.metaharmonizer/data/schema_mapping_eval/`).

To map to **your own** target schema instead of the bundled one, pass
`target_schema_path="your_fields.csv"` (columns: `field_name`,
`is_numeric_field`).

In [ ]:
# The chosen field for each input column:
sm_results[["query", "stage", "method", "match1", "match1_score"]]

## Where to go next

- [`docs/input_formats.md`](../docs/input_formats.md) — full input/output reference, test vs prod mode, env vars.
- [`ontology_mapper_workflow.ipynb`](ontology_mapper_workflow.ipynb) — `test` mode, accuracy scoring, strategy comparison.
- [`schema_mapper_workflow.ipynb`](schema_mapper_workflow.ipynb) — the full Stage 1–4 cascade incl. LLM review.
- [`../README.md`](../README.md) — install, environment variables, engine reference.